<a href="https://colab.research.google.com/github/41371103hjnh/114-1-/blob/main/HW2_%E6%88%90%E7%B8%BE%E4%B8%80%E6%9C%AC%E9%80%9A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [56]:
# Cell 1：安裝套件（Colab 相容版；固定 google-auth / pandas 版本）
%pip install -q \
  gspread \
  google-auth==2.38.0 \
  google-auth-oauthlib \
  pytz \
  pandas==2.2.2 \
  google-generativeai \
  openai \
  --no-warn-conflicts

print("✅ 相容套件安裝完成（含 openai）")


✅ 相容套件安裝完成（含 openai）


In [57]:
# ==========================================
# Cell 2: 基本設定
# ==========================================
import os

# 試算表網址（改成你的 Google Sheet URL）
SHEET_URL = "https://docs.google.com/spreadsheets/d/1eI4Qo9MQaCVInkexGtD6HBaKnvILMJeGwKfHF8nuzBk/edit?usp=sharing"

# 分頁名稱
WORKSHEET_NAME = "工作表一"

# 時區
TIMEZONE = "Asia/Taipei"

print("✅ 基本設定載入完成")
print("SHEET_URL:", "已設定" if SHEET_URL else "未設定")
print("WORKSHEET_NAME:", WORKSHEET_NAME)
print("TIMEZONE:", TIMEZONE)


✅ 基本設定載入完成
SHEET_URL: 已設定
WORKSHEET_NAME: 工作表一
TIMEZONE: Asia/Taipei


In [58]:
# ==========================================
# Cell 3: 匯入＆Gemini 設定（只用 Gemini；不可用就中止）
# ==========================================
from datetime import datetime
import pytz
import pandas as pd
import gspread

# Colab 認證工具
try:
    from google.colab import auth, userdata
except Exception:
    auth = None
    userdata = None

from google.auth import default
import google.generativeai as genai

def _get_api_key():
    # 先從 Colab secrets，再從環境變數
    try:
        if userdata is not None:
            k = userdata.get("GOOGLE_API_KEY") or userdata.get("secretName")
            if k: return k
    except Exception:
        pass
    import os
    return os.environ.get("GOOGLE_API_KEY", "").strip()

GOOGLE_API_KEY = _get_api_key()
USE_GEMINI = False
GEMINI_MODEL = None  # 例如: 'gemini-1.5-flash-8b'

if not GOOGLE_API_KEY:
    raise SystemExit("❌ 未提供 GOOGLE_API_KEY（Colab Secrets 或環境變數），無法產生 AI 註解。")

# 僅使用 Gemini；偏好較常可用的輕量型
genai.configure(api_key=GOOGLE_API_KEY)
candidate_models = [
    "gemini-2.5-flash",

]

# 驗證可用模型（做一次最小請求）
last_error = None
for cand in candidate_models:
    try:
        _ = genai.GenerativeModel(cand).generate_content("ping")
        GEMINI_MODEL = cand
        USE_GEMINI = True
        print(f"✅ Gemini 可用：{GEMINI_MODEL}")
        break
    except Exception as e:
        last_error = e

if not USE_GEMINI:
    raise SystemExit(f"❌ 無法啟用 Gemini（可能是配額/API 未啟用）：{last_error}")



✅ Gemini 可用：gemini-2.5-flash


In [59]:
# ==========================================
# Cell 4: 驗證、連線、開啟試算表／分頁
# ==========================================
if auth is not None:
    print("🔐 進行 Google 帳號驗證…")
    try:
        auth.authenticate_user()
    except Exception as e:
        print("（提示）可能已驗證：", e)

creds, _ = default()
gc = gspread.authorize(creds)

# 開啟試算表
sh = gc.open_by_url(SHEET_URL)

# 欄位標題
_HEADERS = [
    "Batch ID",
    "Timestamp (Asia/Taipei)",
    "Subject",
    "Score",
    "Total (batch)",
    "Average (batch)",
    "AI Note (batch)"
]

# 嘗試取用或建立分頁
try:
    ws = sh.worksheet(WORKSHEET_NAME)
    first_row = ws.row_values(1)
    if first_row != _HEADERS:
        print("ℹ️ 既有分頁標題與此版不同，維持原狀。")
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=WORKSHEET_NAME, rows=1000, cols=len(_HEADERS))
    ws.append_row(_HEADERS)

print("✅ 試算表與分頁就緒")


🔐 進行 Google 帳號驗證…
ℹ️ 既有分頁標題與此版不同，維持原狀。
✅ 試算表與分頁就緒


In [60]:
# ==========================================
# Cell 5: 工具函式（單科一分數；左移欄位；總分/平均/AI 只在最後一列）
# 依賴：
#   - ws 來自 Cell 4（gspread 工作表物件）
#   - USE_GEMINI, GEMINI_MODEL 來自 Cell 3（只用 Gemini）
# ==========================================
from collections import defaultdict
from datetime import datetime
import pytz
import pandas as pd

def input_subject_score_pairs():
    """
    輸入模式（每科一分數）：
    - 依序輸入【科目 → 成績】一組
    - 科目空白 → 結束全部
    - 每個科目只輸入一個分數
    回傳：
      pairs: [{'subject': str, 'score': float}, ...]
      scores: [float, ...]
    """
    pairs, scores = [], []
    i = 1
    print("開始輸入：科目 → 成績；科目空白結束全部（每科只輸入一個分數）。")
    while True:
        subj = input(f"[第 {i} 筆] 科目（空白結束）：").strip()
        if subj == "":
            break
        while True:
            s = input(f"[第 {i} 筆] 成績：").strip()
            if s == "":
                print("⚠️ 成績不可空白，請重新輸入。")
                continue
            try:
                val = float(s)
                break
            except ValueError:
                print("❌ 格式錯誤：請輸入數字（例如 88 或 92.5）。")
        pairs.append({"subject": subj, "score": val})
        scores.append(val)
        i += 1
    return pairs, scores

def compute_total_avg(scores):
    """計算總分與平均值（平均到小數點後 2 位）。"""
    total = sum(scores)
    avg = round(total / len(scores), 2) if scores else 0.0
    return total, f"{avg:.2f}"

def taipei_timestamp(fmt="%Y-%m-%d %H:%M:%S"):
    tz = pytz.timezone("Asia/Taipei")
    return datetime.now(tz).strftime(fmt)

# === 統計與 AI 註解 ===
def _summarize_pairs(pairs):
    """
    回傳 (subject_stats, overall_avg)
    subject_stats = {subj: {"n": 次數, "avg": 平均, "min": 最低, "max": 最高}}
    """
    by_subj = defaultdict(list)
    for p in pairs:
        by_subj[p["subject"]].append(float(p["score"]))
    subject_stats, all_scores = {}, []
    for subj, arr in by_subj.items():
        n = len(arr)
        avg = sum(arr) / n
        mn, mx = min(arr), max(arr)
        subject_stats[subj] = {
            "n": n, "avg": round(avg, 2),
            "min": round(mn, 2), "max": round(mx, 2)
        }
        all_scores.extend(arr)
    overall_avg = round(sum(all_scores) / len(all_scores), 2) if all_scores else 0.0
    return subject_stats, overall_avg

def _stats_table_for_prompt(subject_stats):
    lines = ["科目\t次數\t平均\t最低\t最高"]
    for subj, s in subject_stats.items():
        lines.append(f"{subj}\t{s['n']}\t{s['avg']}\t{s['min']}\t{s['max']}")
    return "\n".join(lines)

def gemini_note_for_pairs(pairs, total, avg_str):
    """
    產生三部分 AI 註解（只用 Gemini；必須成功）：
      1) 整體表現摘要
      2) 單科目讀書建議
      3) 科目強弱分析
    """
    if not (globals().get("USE_GEMINI") and globals().get("GEMINI_MODEL")):
        raise RuntimeError("Gemini 未啟用，無法產生 AI 註解。")

    subject_stats, overall_avg = _summarize_pairs(pairs)
    stats_block = _stats_table_for_prompt(subject_stats)
    brief_pairs = [(p["subject"], float(p["score"])) for p in pairs]

    # 明確要求：不要「好的」等開場白，直接輸出內容
    prompt = f"""
請直接輸出內容，勿加任何開場白或致謝語。

根據以下分數資料，請輸出三個部分：
[資料]
- 分數配對：{brief_pairs}
- 各科統計（tsv）：
{stats_block}
- 總分: {total}, 平均: {avg_str}

[任務]
1) 整體表現摘要（1~2 句）
2) 單科目讀書建議（每科 1~2 句，具體可執行：題型/章節/資源/頻率）
3) 科目強弱分析（依相對表現：強/弱各 1~2 點，附短期行動建議）
""".strip()

    import google.generativeai as genai
    model = genai.GenerativeModel(GEMINI_MODEL)
    resp = model.generate_content(prompt)
    text = (getattr(resp, "text", "") or "").strip()
    if not text:
        raise RuntimeError("Gemini 沒有回傳內容")
    return text

# === 寫入（沒有 Batch ID；總分/平均/AI 只在最後一列） ===
def append_rows_for_pairs(timestamp, pairs, total, avg_str, ai_note=""):
    """
    欄位順序：
    ["Timestamp (Asia/Taipei)", "Subject", "Score", "Total (batch)", "Average (batch)", "AI Note (batch)"]
    """
    if not pairs:
        return []
    rows, n = [], len(pairs)
    for idx, p in enumerate(pairs, start=1):
        s = float(p["score"])
        s_show = int(s) if s.is_integer() else s
        is_last = (idx == n)
        rows.append([
            timestamp,                      # Timestamp
            p["subject"],                   # Subject
            s_show,                         # Score
            total if is_last else "",       # Total (batch) only last row
            avg_str if is_last else "",     # Average (batch) only last row
            ai_note if is_last else ""      # AI Note (batch) only last row
        ])
    ws.append_rows(rows, value_input_option="USER_ENTERED")
    return rows



In [61]:
# ==========================================
# Cell 6: 多科（每科一分數）→ 計算 → 產生 AI → 寫入（總分/平均/AI 只在最後一列）
# 依賴：Cell 4 的 ws、Cell 5 的工具函式、Cell 3 的 Gemini 設定
# ==========================================

# 1) 輸入資料
pairs, scores = input_subject_score_pairs()
if not pairs:
    raise SystemExit("❗ 沒有輸入任何資料，流程結束。")

# 2) 計算總分與平均
total, avg_str = compute_total_avg(scores)

# 3) 產生 AI 摘要（整體摘要、單科建議、強弱分析）
note = gemini_note_for_pairs(pairs, total, avg_str)

# 4) 產生時間戳
ts = taipei_timestamp()

# 5) 寫入 Google Sheet（沒有 Batch ID；總分/平均/AI 只在最後一列）
written = append_rows_for_pairs(ts, pairs, total, avg_str, note)

# 6) 顯示結果（與工作表欄位一致）
df = pd.DataFrame(
    written,
    columns=[
        "Timestamp (Asia/Taipei)",
        "Subject",
        "Score",
        "Total (batch)",
        "Average (batch)",
        "AI Note (batch)"
    ]
)
from IPython.display import display
display(df)

print("\n✅ 已成功寫入！")
print("📌 總分：", total)
print("📌 平均：", avg_str)



開始輸入：科目 → 成績；科目空白結束全部（每科只輸入一個分數）。
[第 1 筆] 科目（空白結束）：國文
[第 1 筆] 成績：95
[第 2 筆] 科目（空白結束）：數學
[第 2 筆] 成績：95
[第 3 筆] 科目（空白結束）：英文
[第 3 筆] 成績：75
[第 4 筆] 科目（空白結束）：


,Timestamp (Asia/Taipei),Subject,Score,Total (batch),Average (batch),AI Note (batch)
0,2025-10-03 17:00:39,國文,95,,,
1,2025-10-03 17:00:39,數學,95,,,
2,2025-10-03 17:00:39,英文,75,265.0,88.33,1) 整體表現摘要\n整體學業表現良好，平均分數為 88.33 分，其中國文與數學表現優異，...



✅ 已成功寫入！
📌 總分： 265.0
📌 平均： 88.33
